# Milestone 1 — Finales Video + robuste Ingestion-Funktion

**Finales MVP-Video:** _Nutrients For Brain Health & Performance_ (Huberman Lab #42) — `E7W4OQfJWdw`

**Ziel:** Aus den M0-Tests eine einzige, wiederverwendbare Funktion `get_transcript(video_id)` bauen, die automatisch zwischen Plan A und Plan B wählt.


## Imports

Wir brauchen dieselben Bausteine wie in M0: die YouTube-Transcript-API, yt-dlp für den Audio-Fallback, und OpenAI für Whisper.


In [1]:
import os
import json

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

import yt_dlp

from dotenv import load_dotenv
from openai import OpenAI

# .env liegt im Projekt-Root, nicht im notebooks/-Ordner -- deswegen der Pfad nach oben (..)
load_dotenv(dotenv_path="../.env")
client = OpenAI()

## download_audio() — Plan-B-Baustein

Holt nur die Audiospur eines Videos (kein Video-Bild), ohne Zeitlimit (anders als in M0, wo wir bewusst nur 3 Minuten getestet haben).

**Offener Punkt für Milestone 2:** Bei sehr langen Videos ohne Untertitel würde Whisper am 25-MB-Limit scheitern. Die Lösung (Audio in Stücke teilen) bauen wir erst, wenn wir sie wirklich brauchen.


In [2]:
def download_audio(video_id: str, output_dir: str = "../audio") -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{video_id}.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_path,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "128",
        }],
        "quiet": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return os.path.join(output_dir, f"{video_id}.mp3")

## transcribe_with_whisper() — schickt Audio an die Whisper-API

Nimmt eine Audiodatei, schickt sie an OpenAI, bekommt Text-Segmente mit Zeitstempeln zurück.


In [3]:
def transcribe_with_whisper(audio_path: str):
    with open(audio_path, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )
    return response.segments

## get_transcript() — die kombinierte Funktion

Das Herzstück von Milestone 1: Plan A zuerst versuchen, bei den zwei bekannten Fehlerfällen automatisch auf Plan B umschalten. Egal welcher Pfad greift, das Rückgabeformat ist immer identisch — eine Liste von Segmenten mit `start`, `end`, `text`.


In [4]:
def get_transcript(video_id: str) -> dict:
    try:
        # Plan A: offizielle YouTube-Untertitel
        raw = YouTubeTranscriptApi().fetch(video_id)
        segments = [
            {"start": s.start, "end": s.start + s.duration, "text": s.text}
            for s in raw
        ]
        source = "youtube_captions"

    except (TranscriptsDisabled, NoTranscriptFound):
        # Plan B: Audio holen + Whisper
        audio_path = download_audio(video_id)
        whisper_segments = transcribe_with_whisper(audio_path)
        segments = [
            {"start": s.start, "end": s.end, "text": s.text}
            for s in whisper_segments
        ]
        source = "whisper_fallback"

    return {
        "video_id": video_id,
        "source": source,   # damit wir später immer wissen, woher die Transcript kam
        "segments": segments,
    }

## Ausführen und Ergebnis speichern

Wir rufen `get_transcript()` mit unserem finalen Video auf und speichern das Ergebnis als JSON-Datei — das ist die Datei, mit der wir in Milestone 2 (Chunking) weiterarbeiten, ohne die Transcript erneut abzurufen.


In [6]:
import os
print(os.path.exists("../data/transcripts"))
print(os.path.isdir("../data/transcripts"))
print(os.path.isfile("../data/transcripts"))

True
False
True


In [7]:
import os

# Die fälschliche Datei löschen (nicht den gewünschten Ordner -- das IST hier die Datei)
os.remove("../data/transcripts")

# Jetzt sollte das Anlegen als Ordner klappen
os.makedirs("../data/transcripts", exist_ok=True)
print("✅ Ordner jetzt korrekt angelegt:", os.path.isdir("../data/transcripts"))

✅ Ordner jetzt korrekt angelegt: True


In [8]:
final_video_id = "E7W4OQfJWdw"

result = get_transcript(final_video_id)

print(f"Quelle: {result['source']}")
print(f"Anzahl Segmente: {len(result['segments'])}\n")
for seg in result["segments"][:5]:
    print(f"[{seg['start']:.1f}s] {seg['text']}")

# Speichern für Milestone 2
os.makedirs("../data/transcripts", exist_ok=True)
output_file = f"../data/transcripts/{final_video_id}.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"\n✅ Gespeichert unter: {output_file}")

Quelle: youtube_captions
Anzahl Segmente: 2502

[0.4s] - Welcome to the Huberman Lab Podcast,
[2.3s] where we discuss science
[3.7s] and science-based tools for everyday life.
[6.0s] [upbeat rock music]
[9.4s] I'm Andrew Huberman,

✅ Gespeichert unter: ../data/transcripts/E7W4OQfJWdw.json
